In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F   
import numpy as np
import tiktoken

In [ ]:
with open('mini_shakespeare.txt') as file:
    file_content = file.read()

alphabet = sorted(list(set(file_content)))
stoi = {char: i for i, char in enumerate(alphabet)}
itos = {i: char for i, char in enumerate(alphabet)}

torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)  

tokenizer = tiktoken.get_encoding('gpt2')

data = torch.tensor(tokenizer.encode(file_content), dtype=torch.long)
cutoff = int(len(data)*0.9)
train = data[:cutoff]
val = data[cutoff:]


device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = 'mps'
print(f'Using {device}')

vocab_size = 50257
block_size = 32
batch_size = 4
embed_dim = 768
num_heads = 12 ## must be divsor of embed_dim
linear_layer = embed_dim*4
blocks = 12
learning_rate = 3e-4

class DataLoaderLite():
    def __init__(self, split):
        self.data = train if split == "train" else val 
        self.batch_num = 0
        pass
    def get_next_batch(self):
        num_tokens = block_size*batch_size
        curr_batch = self.data[self.batch_num*num_tokens: (self.batch_num+1)*num_tokens+1]
        x = curr_batch[:-1].view(batch_size, block_size)
        y = curr_batch[1:].view(batch_size,block_size)
        self.batch_num+=1
        return x.to(device), y.to(device)
    def get_random_batch(self):
        num_tokens = block_size*batch_size
        random_batch = np.random.randint(low=0, high=len(self.data)//(num_tokens)-1)
        curr_batch = self.data[random_batch*num_tokens: (random_batch+1)*num_tokens+1]
        x = curr_batch[:-1].view(batch_size, block_size)
        y = curr_batch[1:].view(batch_size,block_size)
        
        return x.to(device), y.to(device)

class AttentionHead(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.queries = nn.Linear(embed_dim, head_size, bias = False)
        self.keys = nn.Linear(embed_dim, head_size, bias = False)
        self.value_down = nn.Linear(embed_dim, head_size, bias = False)
        self.head_size = head_size
    def forward(self, x):
        q = self.queries(x) ## (B, T, 32)
        k = self.keys(x).transpose(1,2) ## (B, 32, T)
        trans = q @ k /np.sqrt(self.head_size) ## (B, T, T)  
        censored = torch.masked_fill(trans, ~torch.tril(torch.ones((batch_size, block_size, block_size),device=device)).bool(), float("-inf"))
        softmax = torch.softmax(censored, dim=2) ## (B, Querys, Keys)
        attention = softmax @ self.value_down(x) ## (B, T, T) @ (B, T, 32)
        return attention

class MultiHeaded(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([AttentionHead(head_size) for _ in range(num_heads)])
        self.value_up = nn.Linear(head_size*num_heads, embed_dim, bias = False)
        self.value_up.NANOGPT_SCALE_INIT = 1
    def forward(self, x):
        concatenated_value_downs = torch.cat([h(x) for h in self.heads], dim = -1)
        return self.value_up(concatenated_value_downs)

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.layerNorm1 = nn.LayerNorm(embed_dim)
        self.multiheaded_attention = MultiHeaded(num_heads, embed_dim//num_heads)
        self.layerNorm2 = nn.LayerNorm(embed_dim)
        self.linear1 = nn.Linear(embed_dim, linear_layer)
        self.relu = nn.GELU(approximate='tanh')
        self.linear2 = nn.Linear(linear_layer, embed_dim)
        self.linear2.NANOGPT_SCALE_INIT = 1
    def forward(self, x): 
        logits = self.layerNorm1(x)
        logits = self.multiheaded_attention(logits)
        logits = x + logits
  
        linear = self.layerNorm2(logits)
        linear = self.linear1(linear)
        linear = self.relu(linear)
        linear = self.linear2(linear)
        
        logits = logits + linear
        
        return logits 

    
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embeding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = nn.Embedding(block_size, embed_dim)
        self.transformer_block = nn.Sequential(*[TransformerBlock() for i in range(blocks)])
        self.layerNorm = nn.LayerNorm(embed_dim)
        self.final = nn.Linear(embed_dim, vocab_size, bias=False)  # final logits over vocab
        self.token_embeding.weight = self.final.weight
        self.apply(self.__init_weights__)
    def forward(self, x, y = None):
        B, T = x.size()
        logits = self.token_embeding(x) + self.positional_encoding(torch.arange(0, T, dtype=torch.long,device=x.device))
        logits = self.transformer_block(logits)
        logits = self.layerNorm(logits)
        logits = self.final(logits)  # final logits over vocab
        
        
        loss = None
        if y != None:
            B, T, C = logits.shape
            x_reshaped = logits.view(B*T, C)
            y_reshaped = y.view(B*T)
            loss = F.cross_entropy(x_reshaped, y_reshaped)
        return logits, loss
    
    def generate(self, context, max_tokens = 100):
        for i in range(max_tokens):
            x = context[:, -block_size:]
            logits, _ = self(x) ## (B, T, C)
            logits = logits[:, -1, :] ## (B, C) last token
            probs = F.softmax(logits, dim=1)
            next_token = torch.multinomial(probs, num_samples=1) # gets next token
            context = torch.concat((context, next_token[:1]), dim=1)

        return  context

    def __init_weights__(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANOGPT_SCALE_INIT'):
                std *= (2*blocks)**-0.5
            torch.nn.init.normal_(module.weight,mean=0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight,mean=0, std=0.02)




Using cpu


In [ ]:
import time

training_steps = 20
model = Transformer().to(device)
train_dl = DataLoaderLite('train')
val_dl = DataLoaderLite('val')
optim = torch.optim.AdamW(model.parameters(), lr=learning_rate)
total_tokens = 0

torch.set_float32_matmul_precision('high')
for step in range(training_steps):
    start = time.time()

    x, y = train_dl.get_next_batch()
    with torch.autocast(device_type=device, dtype=torch.bfloat16)
    logits, loss = model(x,y)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    optim.step()
    torch.cuda.synchronize()
    elapsed_time = time.time()-start
    tokens_per_sec = (block_size*batch_size)/elapsed_time
    train_loss = loss.item()
    print(f"Step {step}: Train Loss = {train_loss:.4f}, dt = {elapsed_time*1000:.2f}, tok/sec = {tokens_per_sec:.2f}")

    if step % 50 == 0:
        # Training loss
        # Validation loss
        x_val, y_val = val_dl.get_random_batch()
        with torch.no_grad():  
            _, val_loss = model(x_val, y_val)
        val_loss = val_loss.item()
        print(f"Step {step}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


print(loss)
    

KeyboardInterrupt: 

In [ ]:
# start with a context of just one token, e.g. the index for "H"
start = torch.tensor([tokenizer.encode("".join([" "]*32))], dtype=torch.long)

# generate 100 new tokens
out = model.generate(start, max_tokens=100)

# convert indices back to characters
generated_text = ''.join(tokenizer.decode(out))
print(generated_text)

                                froitt dimf, thesenss! Hrures, ghere fate! in fom rage Inconer olldtiottiungl seaid me botemo fatory
